# temperature — 같은 질문에 같은 대답 받기

`1_1_openai` 에서 배운 호출에 **옵션 하나**를 더합니다.
같은 질문을 여러 번 보내도 **매번 같은 답이 나오게** 만드는 방법입니다.

## 이 노트북의 흐름

| 단계 | 내용 |
|---|---|
| **Step 1~2** | 준비 — 라이브러리, API 키, 클라이언트 |
| **Step 3~4** | 실험 준비 — 질문과 "여러 번 묻기" 함수 |
| **Step 5** | temperature 를 높게 — 답이 매번 달라진다 |
| **Step 6** | temperature 를 0 으로 — 답이 거의 같아진다 |
| **Step 7** | `seed` 까지 — 재현성을 더 높인다 |
| **Step 8** | 주의 — temperature 를 못 쓰는 모델 |

> **이 노트북에서 기억할 한 가지**
> `temperature=0` 과 `seed` 를 주면 같은 질문에 **같은 답**을 받을 수 있다.
> 단, 100% 보장은 아니고 **"거의 항상"** 이다.

## temperature 가 뭔가

LLM 은 다음 단어를 **확률로 고릅니다.**
"오늘 날씨가 ___" 다음에 올 단어 후보가 이렇게 있다고 해 봅시다.

| 후보 | 확률 |
|---|---|
| 좋다 | 60% |
| 맑다 | 25% |
| 흐리다 | 10% |
| 덥다 | 5% |

`temperature` 는 **이 확률표를 얼마나 '뭉개느냐'** 를 정합니다.

| temperature | 하는 일 | 결과 |
|---|---|---|
| `0` | 거의 항상 1등(좋다)만 고른다 | **매번 같은 답** — 정답·분류·요약에 |
| `1` (기본값) | 확률대로 고른다 | 적당히 다양함 |
| `1.5 ~ 2` | 낮은 후보도 자주 뽑힌다 | 창의적이지만 엉뚱해질 수 있음 |

> 이름이 '온도'인 이유: 온도가 높으면 분자가 마구 움직이듯,
> 값이 높을수록 **답이 이리저리 튑니다.**

## Step 1. 준비 — 라이브러리와 API 키

`1_1_openai` 의 Step 1~2 와 같습니다.
`load_dotenv()` 는 현재 폴더에서 시작해 **위쪽 폴더로 올라가며** `.env` 를 찾습니다.

In [ ]:
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')

if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")

## Step 2. 클라이언트와 모델 정하기

```python
MODEL = "gpt-4.1-mini"
```

이번 노트북은 **`gpt-4.1-mini`** 를 씁니다.
`1_1_openai` Step 5 의 `gpt-5-nano` 는 **temperature 를 바꿀 수 없는 모델**이기 때문입니다.
(이유는 Step 8 에서 직접 확인합니다.)

모델 이름을 변수 하나로 빼 두면, 나중에 바꿀 때 **한 곳만 고치면 됩니다.**

In [ ]:
openai = OpenAI()
MODEL = "gpt-4.1-mini"

## Step 3. 실험용 질문

답이 **여러 가지로 나올 수 있는 질문**이어야 차이가 잘 보입니다.
"1+1은?" 같은 질문은 temperature 와 상관없이 늘 "2"라서 실험이 안 됩니다.

그래서 **짧은 창작**을 시킵니다. 한 문장으로 제한해 비교하기 쉽게 합니다.

In [ ]:
messages = [
    {"role": "system", "content": "당신은 카피라이터입니다. 반드시 한 문장으로만 답하세요."},
    {"role": "user",   "content": "커피숍 이름을 하나 지어 주고, 왜 그렇게 지었는지 알려 주세요."},
]
messages

## Step 4. "같은 질문 여러 번 묻기" 함수

같은 `messages` 를 `times` 번 보내고 답을 목록으로 모읍니다.

```python
def ask_many(times, **options):
```

`**options` 는 **이름 붙은 인자를 몽땅 받아서 그대로 넘겨주는** 문법입니다.
`ask_many(3, temperature=0)` 이라고 부르면 `options` 는 `{"temperature": 0}` 이 되고,
`create(..., **options)` 에서 `temperature=0` 으로 풀려 들어갑니다.

마지막에 `len(set(answers))` 로 **서로 다른 답이 몇 종류인지** 셉니다.
`set` 은 중복을 없애므로, 모든 답이 같다면 `1` 이 나옵니다.

In [ ]:
def ask_many(times, **options):
    answers = []
    for i in range(times):
        response = openai.chat.completions.create(model=MODEL, messages=messages, **options)
        answer = response.choices[0].message.content
        answers.append(answer)
        print(f"[{i+1}] {answer}")
    print(f"\n서로 다른 답: {len(set(answers))}개 / {times}번 중")
    return answers

## Step 5. temperature 를 높게 — 매번 다른 답

```python
ask_many(3, temperature=1.5)
```

같은 질문을 3번 보냈는데 **커피숍 이름이 매번 다르게** 나올 것입니다.
아래 "서로 다른 답" 이 `3개` 에 가깝게 나옵니다.

In [ ]:
high = ask_many(3, temperature=1.5)

## Step 6. temperature 를 0 으로 — 거의 같은 답

```python
ask_many(3, temperature=0)
```

이번엔 매번 **확률 1등 단어만** 고르므로 같은 답이 반복됩니다.
"서로 다른 답" 이 `1개` 로 나오면 성공입니다.

> **그런데 가끔 2개가 나온다면?**
> 고장 난 게 아닙니다. 서버의 GPU 계산은 아주 미세한 오차가 있어서,
> 1등과 2등 확률이 거의 같을 때 **드물게 순서가 뒤집힙니다.**
> 한 단어만 달라져도 그 뒤 문장은 다른 길로 갑니다.
> 그래서 다음 Step 의 `seed` 를 함께 씁니다.

In [ ]:
zero = ask_many(3, temperature=0)

## Step 7. `seed` — 재현성을 한 단계 더

```python
ask_many(3, temperature=0, seed=42)
```

`seed` 는 **무작위 뽑기의 출발점을 고정**하는 숫자입니다.
같은 seed 를 주면 OpenAI 가 **최대한 같은 계산 경로**로 답을 만들어 줍니다.
숫자 자체(42)에는 의미가 없습니다. **매번 같은 숫자를 쓰는 것**이 중요합니다.

### `system_fingerprint` — 서버가 바뀌었는지 확인

seed 를 줘도 OpenAI 가 **서버 쪽 설정을 업데이트하면** 답이 달라질 수 있습니다.
응답의 `system_fingerprint` 가 그 설정 버전입니다.

| 상황 | 결과 |
|---|---|
| seed 같음 + fingerprint 같음 | 거의 확실히 같은 답 |
| seed 같음 + fingerprint 다름 | 서버가 바뀌어서 **답이 달라질 수 있음** |

> 공식 문서도 seed 는 **"최선을 다한다(best effort)"** 이지 보장은 아니라고 말합니다.

In [ ]:
seeded = ask_many(3, temperature=0, seed=42)

In [ ]:
response = openai.chat.completions.create(model=MODEL, messages=messages, temperature=0, seed=42)
print("답변       :", response.choices[0].message.content)
print("fingerprint:", response.system_fingerprint)

## Step 8. 주의 — temperature 를 못 쓰는 모델

`gpt-5-nano` 같은 **추론(reasoning) 모델**은 답하기 전에 속으로 여러 번 생각하는 방식이라
temperature 를 **기본값 1 로 고정**해 두었습니다. 다른 값을 주면 오류가 납니다.

아래 셀은 **일부러 오류를 내 보는** 셀입니다.
`try / except` 로 감싸서 노트북이 멈추지 않고 오류 메시지만 출력합니다.

In [ ]:
try:
    openai.chat.completions.create(model="gpt-5-nano", messages=messages, temperature=0)
except Exception as e:
    print(type(e).__name__, ":", e)

> 추론 모델에서 답을 일정하게 받고 싶다면 temperature 대신
> **`system` 프롬프트로 형식을 엄격하게 정해 주는 방법**을 씁니다.
> 예: "반드시 '이름: ___ / 이유: ___' 형식으로만 답하세요."

---

## 정리

```
같은 답이 필요하다  ->  temperature=0  +  seed=고정숫자
다양한 답이 필요하다 ->  temperature 를 1 이상으로
```

### 꼭 기억할 것

| | 내용 |
|---|---|
| **temperature 는 다양성 조절 손잡이** | 낮을수록 일정, 높을수록 다양 |
| **`0` 이면 거의 같은 답** | 정답·분류·요약·데이터 추출에 쓴다 |
| **`seed` 로 한 번 더 고정** | 그래도 100% 보장은 아니다 |
| **`system_fingerprint` 확인** | 서버가 바뀌면 답이 달라질 수 있다 |
| **추론 모델은 못 바꾼다** | `gpt-5` 계열은 기본값 1 로 고정 |

### 직접 해 보기

1. Step 5 에서 `temperature=2` 로 올려 보세요. 답이 얼마나 이상해지나요?
2. Step 6 에서 `times` 를 10 으로 늘려 보세요. "서로 다른 답" 이 몇 개 나오나요?
3. Step 7 에서 `seed` 를 42 가 아닌 다른 숫자로 바꿔 보세요. 답이 바뀌나요?
4. Step 3 의 질문을 "대한민국의 수도는?" 으로 바꾸고 Step 5 를 다시 해 보세요.
   왜 temperature 가 높아도 답이 거의 같을까요?